# 04B — Contrôle négatif de couverture par TPE catégoriel

Ce notebook conserve le nom historique 04B/Optuna, mais corrige son rôle scientifique. **03B reste l’unique autorité de sélection** et 04A sa référence auditée. 04B ne recherche, ne réajuste et ne sélectionne aucun modèle.

Optuna reçoit un unique paramètre catégoriel `model_id` et consulte les objectifs déjà calculés en 03B. Comme cet identifiant ne porte aucune géométrie d’hyperparamètres, TPE ne peut pas généraliser d’un modèle vers un autre : l’étude est donc un contrôle négatif diagnostique de couverture sous budget, comparé à l’espérance d’un tirage uniforme avec remise. Ses résultats sont interdits pour toute sélection aval.

## A — Initialisation et chemins centralisés

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import optuna
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
if (current_dir / "src").is_dir():
    project_root = current_dir
elif (current_dir.parent / "src").is_dir():
    project_root = current_dir.parent
else:
    raise RuntimeError("Launch 04B from the repository or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import experiment_config as expcfg
from src.protocol_governance import (
    sha256_file,
    sha256_payload,
    verify_frozen_protocol,
    resolve_protocol_for_execution,
)
from src.utils import load_parquet, save_parquet
from src.workflows.protocol_audit import assert_no_forbidden_score_columns
from src.workflows.simca_budgeted_search_audit import (
    run_categorical_tpe_coverage_benchmark,
)
from src.workflows.simca_calibration_registry import (
    validate_internal_calibration_manifest,
)

In [2]:
results_tag = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
protocol_dir = project_root.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
input_dir_03b = (
    project_root
    / "results"
    / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{results_tag}"
)
input_dir_04a = (
    project_root
    / "results"
    / f"{expcfg.SIMCA_GRID_SEARCH_RESULTS_DIR_PREFIX}_{results_tag}"
)
output_dir = (
    project_root
    / "results"
    / f"{expcfg.SIMCA_OPTUNA_RESULTS_DIR_PREFIX}_{results_tag}"
)
output_dir.mkdir(parents=True, exist_ok=True)

input_paths_03b = {
    key: input_dir_03b / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in expcfg.SIMCA_OPTUNA_REQUIRED_03B_ARTIFACTS
}
input_manifest_path_03b = (
    input_dir_03b
    / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["checkpoint_manifest"]
)
input_paths_04a = {
    key: input_dir_04a / expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES[key]
    for key in expcfg.SIMCA_OPTUNA_REQUIRED_04A_ARTIFACTS
}
output_paths = {
    key: output_dir / filename
    for key, filename in expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES.items()
}

print("03B:", input_dir_03b)
print("04A:", input_dir_04a)
print("04B:", output_dir)

03B: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03B_internal_calibration_8tracks_v5_px_qc_v1
04A: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_8tracks_v5_px_qc_v1
04B: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v5_px_qc_v1


## B — Vérification bloquante des contrats 03B–04A

In [3]:
# verify_frozen_protocol(protocol_dir, strict=True)
# protocol_lock = json.loads(
#     (protocol_dir / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(
#         encoding="utf-8"
#     )
# )
# protocol_hash = str(protocol_lock["lock_sha256"])
protocol_checks_df, protocol_hash = (
    resolve_protocol_for_execution(
        protocol_dir,
        upstream_manifest_path=input_manifest_path_03b,
    )
)
if not input_manifest_path_03b.is_file():
    raise FileNotFoundError(input_manifest_path_03b)
manifest_03b = json.loads(input_manifest_path_03b.read_text(encoding="utf-8"))
validate_internal_calibration_manifest(
    manifest_03b,
    input_paths_03b,
    required_artifacts=expcfg.SIMCA_OPTUNA_REQUIRED_03B_ARTIFACTS,
    protocol_hash=protocol_hash,
)
artifact_hashes_03b = {
    str(entry["name"]): str(entry["sha256"])
    for entry in manifest_03b["artifacts"]
}

manifest_04a = json.loads(
    input_paths_04a["audit_manifest"].read_text(encoding="utf-8")
)
if str(manifest_04a["protocol_hash"]) != protocol_hash:
    raise RuntimeError("04A and the frozen protocol have different hashes.")
if manifest_04a["input_03b_manifest_sha256"] != sha256_file(input_manifest_path_03b):
    raise RuntimeError("04A was not built from the current 03B manifest.")
if manifest_04a["selection_authority"] != "03B_selected_models":
    raise RuntimeError("04A does not preserve the 03B selection authority.")
if bool(manifest_04a["selection_mutated"]):
    raise RuntimeError("04A reports a mutation of selected_models.")
if manifest_04a.get("spatial_selection_scope") != expcfg.SPATIAL_CALIBRATION_SELECTION_SCOPE:
    raise RuntimeError("04A does not propagate the within-track 03C spatial contract.")
if manifest_04a.get("spatial_selection_policy") != expcfg.SPATIAL_CALIBRATION_SELECTION_POLICY:
    raise RuntimeError("04A propagates an unexpected spatial selection policy.")
reference_entry = manifest_04a["output_artifacts"]["model_reference"]
if reference_entry["sha256"] != sha256_file(input_paths_04a["model_reference"]):
    raise RuntimeError("04A model-reference hash mismatch.")

track_contracts = load_parquet(input_paths_03b["track_contracts"])
model_catalog = load_parquet(input_paths_03b["model_catalog"])
model_metrics = load_parquet(input_paths_03b["model_metrics"])
selected_models = load_parquet(input_paths_03b["selected_models"])
model_reference = load_parquet(input_paths_04a["model_reference"])
if set(model_reference["model_id"].astype(str)) != set(
    selected_models["model_id"].astype(str)
):
    raise RuntimeError("04A and 03B selected-model sets differ.")
if set(track_contracts["track_id"].astype(str)) != {
    f"E{index}" for index in range(1, 9)
}:
    raise RuntimeError("The 03B contract must contain exactly E1-E8.")

## C — Benchmark TPE catégoriel sans refit

L’univers contient uniquement les modèles dont tous les objectifs Pareto 03B propres au track sont finis. Les objectifs sont consultés dans `model_metrics.parquet` mais ne sont pas recopiés : chaque trial reste joignable par `model_id`. La seule boucle séquentielle est celle imposée par l’optimiseur ; préparation, contrôles et synthèse sont vectorisés.

In [4]:
if not expcfg.SIMCA_OPTUNA_REUSE_INTERNAL_METRICS:
    raise RuntimeError("04B must reuse the frozen 03B metrics.")
if expcfg.SIMCA_OPTUNA_RUN:
    benchmark_outputs = run_categorical_tpe_coverage_benchmark(
        model_catalog=model_catalog,
        model_metrics=model_metrics,
        selected_models=selected_models,
        track_contracts=track_contracts,
        model_reference=model_reference,
    )
else:
    benchmark_outputs = {
        key: load_parquet(output_paths[key])
        for key in ("sampled_models", "search_efficiency")
    }

sampled_models = benchmark_outputs["sampled_models"]
search_efficiency = benchmark_outputs["search_efficiency"]
if tuple(sampled_models.columns) != expcfg.SIMCA_OPTUNA_BENCHMARK_SAMPLE_COLUMNS:
    raise RuntimeError("Unexpected 04B sampled-model schema.")
if tuple(search_efficiency.columns) != expcfg.SIMCA_OPTUNA_BENCHMARK_SUMMARY_COLUMNS:
    raise RuntimeError("Unexpected 04B search-efficiency schema.")
if sampled_models.duplicated(["track_id", "trial_number"]).any():
    raise RuntimeError("04B trial sequence keys are not unique.")
if set(search_efficiency["track_id"].astype(str)) != {
    f"E{index}" for index in range(1, 9)
}:
    raise RuntimeError("04B summary must retain tracks E1-E8.")

display(search_efficiency)
display(assert_no_forbidden_score_columns(benchmark_outputs))

,track_id,downstream_status,n_evaluable_models,n_selected_reference_models,trial_budget,n_unique_models_sampled,duplicate_trial_rate,model_coverage_rate,n_selected_reference_recovered,selected_reference_recall,uniform_expected_selected_recall,recall_delta_vs_uniform
0,E1,supported,215,2,100,87,0.13,0.404651,0,0.0,0.372619,-0.372619
1,E2,supported,2094,8,100,93,0.07,0.044413,0,0.0,0.046644,-0.046644
2,E3,diagnostic_only,3,1,100,3,0.97,1.000000,1,1.0,1.000000,0.000000
3,E4,diagnostic_only,3,1,100,3,0.97,1.000000,1,1.0,1.000000,0.000000
4,E5,supported,2082,1,100,93,0.07,0.044669,0,0.0,0.046907,-0.046907
5,E6,supported,7616,17,100,99,0.01,0.012999,0,0.0,0.013045,-0.013045
6,E7,supported,60,1,100,60,0.40,1.000000,1,1.0,0.813759,0.186241
7,E8,supported,2441,7,100,94,0.06,0.038509,0,0.0,0.040147,-0.040147


,table,n_columns,forbidden_score_columns,score_free
0,sampled_models,5,,True
1,search_efficiency,12,,True


## D — Interprétation scientifique

Le rappel observé mesure seulement la visite des références 03B. Une valeur faible ne remet pas en cause ces modèles : elle montre qu’un budget de 100 évaluations et une catégorie sans structure ne permettent pas de les retrouver de façon fiable. Les tracks `diagnostic_only` restent affichés mais ne sont pas promus.

In [5]:
supported_summary = search_efficiency.loc[
    search_efficiency["downstream_status"].eq("supported")
]
interpretation = pd.DataFrame(
    {
        "n_supported_tracks": [int(len(supported_summary))],
        "n_supported_tracks_with_reference_recovery": [
            int(supported_summary["n_selected_reference_recovered"].gt(0).sum())
        ],
        "mean_supported_reference_recall": [
            float(supported_summary["selected_reference_recall"].mean())
        ],
        "mean_uniform_expected_recall": [
            float(supported_summary["uniform_expected_selected_recall"].mean())
        ],
    }
)
display(interpretation)

,n_supported_tracks,n_supported_tracks_with_reference_recovery,mean_supported_reference_recall,mean_uniform_expected_recall
0,6,1,0.166667,0.222187


## E — Persistance compacte et provenance

In [6]:
if expcfg.SIMCA_OPTUNA_RUN:
    save_parquet(sampled_models, output_paths["sampled_models"], optimize=False)
    save_parquet(search_efficiency, output_paths["search_efficiency"], optimize=False)

input_hashes = {
    **{
        f"03B.{key}": artifact_hashes_03b[key]
        for key in expcfg.SIMCA_OPTUNA_REQUIRED_03B_ARTIFACTS
    },
    "04A.model_reference": str(reference_entry["sha256"]),
    "04A.audit_manifest": sha256_file(input_paths_04a["audit_manifest"]),
}
plan_payload = {
    "rule_version": expcfg.SIMCA_OPTUNA_BENCHMARK_RULE_VERSION,
    "benchmark_role": expcfg.SIMCA_OPTUNA_BENCHMARK_ROLE,
    "parameter": expcfg.SIMCA_OPTUNA_BENCHMARK_PARAMETER,
    "trial_budget_per_track": int(expcfg.SIMCA_OPTUNA_N_TRIALS_PER_TRACK),
    "n_startup_trials": int(expcfg.SIMCA_OPTUNA_N_STARTUP_TRIALS),
    "random_state": int(expcfg.SIMCA_OPTUNA_RANDOM_STATE),
    "sampler": expcfg.SIMCA_OPTUNA_SAMPLER_NAME,
    "objectives": expcfg.INTERNAL_CALIBRATION_PARETO_OBJECTIVES,
    "input_sha256": input_hashes,
}
benchmark_plan_hash = sha256_payload(plan_payload)
output_artifacts = {
    key: {
        "path": str(output_paths[key]),
        "row_count": int(len(table)),
        "columns": list(table.columns),
        "sha256": sha256_file(output_paths[key]),
    }
    for key, table in benchmark_outputs.items()
}
audit_manifest = {
    "notebook": "04B_simca_optuna_search",
    "protocol_task_ids": [29, 30],
    "protocol_version": str(expcfg.PROTOCOL_VERSION),
    "schema_version": str(expcfg.RESULTS_SCHEMA_VERSION),
    "protocol_hash": protocol_hash,
    "benchmark_plan_hash": benchmark_plan_hash,
    "benchmark_role": expcfg.SIMCA_OPTUNA_BENCHMARK_ROLE,
    "selection_authority": "03B_selected_models",
    "selection_mutated": False,
    "model_refit": False,
    "threshold_resuggestion": False,
    "downstream_selection_use": "forbidden",
    "suggested_parameters": [expcfg.SIMCA_OPTUNA_BENCHMARK_PARAMETER],
    "categorical_limitation": (
        "model_id has no hyperparameter geometry; TPE coverage is a negative "
        "control and not evidence of model-search efficiency"
    ),
    "uniform_reference": "sampling_with_replacement",
    "uniform_expected_recall_formula": "1-(1-1/N)^B",
    "weighted_score_used": False,
    "inherited_spatial_selection_scope": str(
        manifest_04a["spatial_selection_scope"]
    ),
    "inherited_spatial_selection_policy": str(
        manifest_04a["spatial_selection_policy"]
    ),
    "input_03b_manifest_sha256": sha256_file(input_manifest_path_03b),
    "input_04a_manifest_sha256": sha256_file(
        input_paths_04a["audit_manifest"]
    ),
    "input_sha256": input_hashes,
    "output_artifacts": output_artifacts,
    "counts": {
        "tracks": int(len(search_efficiency)),
        "trials": int(len(sampled_models)),
        "evaluable_models": int(
            search_efficiency["n_evaluable_models"].sum()
        ),
        "selected_reference_models": int(
            search_efficiency["n_selected_reference_models"].sum()
        ),
        "selected_reference_models_recovered": int(
            search_efficiency["n_selected_reference_recovered"].sum()
        ),
    },
    "software": {"optuna": optuna.__version__},
}
audit_manifest["manifest_payload_sha256"] = sha256_payload(audit_manifest)
output_paths["audit_manifest"].write_text(
    json.dumps(audit_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved outputs:")
for path in output_paths.values():
    print(" -", path)

Saved outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v5_px_qc_v1\categorical_tpe_sampled_models.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v5_px_qc_v1\categorical_tpe_coverage.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_optuna_search_8tracks_v5_px_qc_v1\audit_manifest.json


## Lecture des sorties

- `categorical_tpe_sampled_models.parquet` : séquence compacte `(track_id, trial_number)` et `model_id` déjà défini en 03B ; les objectifs restent dans `model_metrics.parquet`.
- `categorical_tpe_coverage.parquet` : une ligne par track avec couverture, répétitions et rappel des références 03B comparé au nul uniforme.
- `audit_manifest.json` : provenance et interdiction explicite d’utiliser 04B pour modifier la sélection.